Diffusers and pipeline Code

In [1]:
from diffusers import StableDiffusion3Pipeline
import torch
from config import LAM_VALUES, TSR_DIR, PT_TSR_DIR, PROMPTS_FILE, MODEL_CACHE, SEED, TSR_SIGMA, SWAP_ALGORITHM, N_INF_STEPS, GUIDANCE_SCALE

    # ── Load model once ───────────────────────────────────────────────────────────
pipe = StableDiffusion3Pipeline.from_pretrained(
	"stabilityai/stable-diffusion-3-medium-diffusers",
	torch_dtype=torch.float16,
	cache_dir=MODEL_CACHE,
)
pipe = pipe.to("cuda")
pipe.set_progress_bar_config(disable=True)

Loading pipeline components...:   0%|          | 0/9 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Imports, configs

In [2]:
from pathlib import Path
import gc
import pandas as pd
from tqdm import tqdm
import shutil

TSR_DIR     = Path("/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/images/new_tsr_samples_tester")
PT_TSR_DIR  = Path("/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/images/new_pt_samples_tester")

REPLICA_EXCHANGE = True
LAM_VALUES = [2, 1.15, 1.01, 1.0]
INDEX_UNTIL = 10

gc.collect()
torch.cuda.empty_cache()
    

In [3]:
# if PT_TSR_DIR.exists():
#     shutil.rmtree(PT_TSR_DIR)


Sample!

In [ ]:
# ── Load prompts once ─────────────────────────────────────────────────────────
prompts = pd.read_csv(PROMPTS_FILE, usecols=["text"], nrows=INDEX_UNTIL)["text"].tolist()
print(f"Loaded {len(prompts)} prompts")


replica_exchanges = [True, False]

lam_dirs = {}
for re in replica_exchanges:
	base = PT_TSR_DIR if re else TSR_DIR
	lam_dirs[re] = {l: base / f"lam{l:.3f}".replace(".", "p") for l in LAM_VALUES}
	for d in lam_dirs[re].values():
		d.mkdir(parents=True, exist_ok=True)

# ── Sweep ─────────────────────────────────────────────────────────────────────
for idx, prompt in enumerate(prompts):
	
	for replica_exchange in replica_exchanges:
	
		for tsr_lam in tqdm(LAM_VALUES, desc=f"idx={idx} re={replica_exchange}"):

			output_dir = lam_dirs[replica_exchange][tsr_lam]

			if (output_dir / f"{idx:05d}.png").exists():
				continue

			generator = torch.Generator(device="cuda").manual_seed(SEED)

			images = pipe(
				prompt,
				negative_prompt="",
				num_inference_steps=N_INF_STEPS,
				guidance_scale=GUIDANCE_SCALE,
				tsr_lam=tsr_lam,
				tsr_sigma=TSR_SIGMA,
				replica_exchange=replica_exchange,
				swap_algorithm=SWAP_ALGORITHM,
				generator=generator,
			).images

			out_path = output_dir / f"{idx:05d}.png"
			images[0].save(out_path, icc_profile=None)


			del images
		torch.cuda.empty_cache()

print("\n All k values complete.")

Loaded 10 prompts


idx=0 re=True:   0%|          | 0/4 [00:00<?, ?it/s]

 We tsr by 1.15 with replica exchange True
 fs 0.0 between lams s 0.86962890625 and lam t 1.150390625 temp s 1.0 temp t 1.0
Time 1000.00 swap btwn source 0.87 and target 1.15 accept 1.000 std 0.984
 fs -0.00018590160470921546 between lams s 0.86962890625 and lam t 1.150390625 temp s 1.000701665878296 temp t 0.9991917610168457
Time 975.98 swap btwn source 0.87 and target 1.15 accept 1.000 std 0.959
 fs -0.0008617350249551237 between lams s 0.86962890625 and lam t 1.150390625 temp s 1.0032198429107666 temp t 0.996311366558075
Time 949.53 swap btwn source 0.87 and target 1.15 accept 0.999 std 0.935
 fs -0.002349973190575838 between lams s 0.86962890625 and lam t 1.150390625 temp s 1.0082885026931763 temp t 0.9906063675880432
Time 920.28 swap btwn source 0.87 and target 1.15 accept 0.997 std 0.911
 fs -0.005208190996199846 between lams s 0.86962890625 and lam t 1.150390625 temp s 1.0166891813278198 temp t 0.9814159274101257
Time 887.74 swap btwn source 0.87 and target 1.15 accept 0.994 std

idx=1 re=True:   0%|          | 0/4 [00:00<?, ?it/s]

 We tsr by 1.15 with replica exchange True
 fs 0.0 between lams s 0.86962890625 and lam t 1.150390625 temp s 1.0 temp t 1.0
Time 1000.00 swap btwn source 0.87 and target 1.15 accept 1.000 std 0.985
 fs -0.00015698332572355866 between lams s 0.86962890625 and lam t 1.150390625 temp s 1.000701665878296 temp t 0.9991917610168457
Time 975.98 swap btwn source 0.87 and target 1.15 accept 1.000 std 0.959
 fs -0.0009137733723036945 between lams s 0.86962890625 and lam t 1.150390625 temp s 1.0032198429107666 temp t 0.996311366558075
Time 949.53 swap btwn source 0.87 and target 1.15 accept 0.999 std 0.935
 fs -0.0017440298106521368 between lams s 0.86962890625 and lam t 1.150390625 temp s 1.0082885026931763 temp t 0.9906063675880432
Time 920.28 swap btwn source 0.87 and target 1.15 accept 0.998 std 0.910
 fs -0.004658044315874577 between lams s 0.86962890625 and lam t 1.150390625 temp s 1.0166891813278198 temp t 0.9814159274101257
Time 887.74 swap btwn source 0.87 and target 1.15 accept 0.995 st

...

 We tsr by 2.00 with replica exchange True
 fs 0.0 between lams s 0.5 and lam t 2.0 temp s 1.0 temp t 1.0
Time 1000.00 swap btwn source 0.50 and target 2.00 accept 1.000 std 0.985
 fs -0.000492171966470778 between lams s 0.5 and lam t 2.0 temp s 1.0026966333389282 temp t 0.9946503043174744
Time 975.98 swap btwn source 0.50 and target 2.00 accept 0.999 std 0.961
 fs -0.0026023010723292828 between lams s 0.5 and lam t 2.0 temp s 1.0124624967575073 temp t 0.9759734869003296
Time 949.53 swap btwn source 0.50 and target 2.00 accept 0.997 std 0.935
 fs -0.006361029110848904 between lams s 0.5 and lam t 2.0 temp s 1.03255295753479 temp t 0.9406864047050476
Time 920.28 swap btwn source 0.50 and target 2.00 accept 0.994 std 0.907
 fs -0.01407591626048088 between lams s 0.5 and lam t 2.0 temp s 1.067185401916504 temp t 0.8881691694259644
Time 887.74 swap btwn source 0.50 and target 2.00 accept 0.986 std 0.878
 fs -0.026068124920129776 between lams s 0.5 and lam t 2.0 temp s 1.1202843189239502 te

idx=2 re=True:  25%|██▌       | 1/4 [00:28<01:26, 28.95s/it]

 We tsr by 1.15 with replica exchange True
 fs 0.0 between lams s 0.86962890625 and lam t 1.150390625 temp s 1.0 temp t 1.0
Time 1000.00 swap btwn source 0.87 and target 1.15 accept 1.000 std 0.985
 fs -8.94616823643446e-05 between lams s 0.86962890625 and lam t 1.150390625 temp s 1.000701665878296 temp t 0.9991917610168457
Time 975.98 swap btwn source 0.87 and target 1.15 accept 1.000 std 0.961
 fs -0.0004695181269198656 between lams s 0.86962890625 and lam t 1.150390625 temp s 1.0032198429107666 temp t 0.996311366558075
Time 949.53 swap btwn source 0.87 and target 1.15 accept 0.999 std 0.935
 fs -0.0012022267328575253 between lams s 0.86962890625 and lam t 1.150390625 temp s 1.0082885026931763 temp t 0.9906063675880432
Time 920.28 swap btwn source 0.87 and target 1.15 accept 0.999 std 0.907
 fs -0.0027840747497975826 between lams s 0.86962890625 and lam t 1.150390625 temp s 1.0166891813278198 temp t 0.9814159274101257
Time 887.74 swap btwn source 0.87 and target 1.15 accept 0.997 std

idx=2 re=True:  50%|█████     | 2/4 [00:57<00:57, 28.94s/it]

 We tsr by 1.01 with replica exchange True
 fs 0.0 between lams s 0.990234375 and lam t 1.009765625 temp s 1.0 temp t 1.0
Time 1000.00 swap btwn source 0.99 and target 1.01 accept 1.000 std 0.985
 fs 0.0 between lams s 0.990234375 and lam t 1.009765625 temp s 1.000052571296692 temp t 0.9999474883079529
Time 975.98 swap btwn source 0.99 and target 1.01 accept 1.000 std 0.961


Fid computation

In [ ]:
from fid import compute_sweep
# from config import LAM_VALUES, TSR_DIR, PT_TSR_DIR, PROMPTS_FILE, MODEL_CACHE, SEED, TSR_SIGMA, SWAP_ALGORITHM, N_INF_STEPS, GUIDANCE_SCALE
# INDEX_UNTIL = 100

compute_sweep(
	lam_values=LAM_VALUES,
	replica_exchanges=[True, False],
	device="cuda",
	target_indices=None,
	index_until = INDEX_UNTIL,
	pt_sr_dir = PT_TSR_DIR,
	tsr_dir = TSR_DIR,
)